<br>

# Municípios e Comarcas

Por meio do _site_ [ListaTelefonica](https://www.tjsp.jus.br/ListaTelefonica) foi possível relevar as APIs que operam e fazer diversas pesquisas.

<br>

Michel Metran\
Data: 18.06.2024\
Atualizado em: 18.06.2024


In [ ]:
import pandas as pd

from tjsp_unidades.api_tel import ListarUnidades
from tjsp_unidades.paths import output_path_tab
from tjsp_unidades.small_functions import adjust_columns

In [ ]:
import concurrent.futures

<br>

---

## Lista de Unidades

Uma vez com a lista de todos os municípios e seus respectivos códigos, foi possível obter as informações dos fóruns por meio de outro método `POST`.


In [ ]:
lu = ListarUnidades()
print(lu.df_com)

In [ ]:
# Apenas Exemplo da Função!
lu.get_lista_unidades_tjsp(id_municipio_tjsp=6504)

<br>

Uma vez com a função que obter os dados das Unidades e, PRINCIPALMENTE, das comarcas!
Foi possível iterar pelos 645 municípios para pegar as informações!

> IMPORTANTE: FUNÇÃO LEVA UNS 4 MINUTOS PARA RODAR!


In [ ]:
# Read Data
df_tjsp_mun = pd.read_csv(
    filepath_or_buffer=output_path_tab / "Municipios.csv",
)

# Results
df_tjsp_mun.info()
df_tjsp_mun.head()

In [ ]:
lu.get_comarcas(df_mun=df_tjsp_mun)

Uma vez que temos a tabela de municípios e a tabela de comarcas/unidades, tratamos e juntamos


In [ ]:
# Display
display(df_tjsp_mun.head(2))

In [ ]:
display(lu.unidades.info())
display(lu.unidades.head(2))

In [ ]:
display(lu.comarcas.info())
display(lu.comarcas.head(2))

In [ ]:
filename = "Unidades, Municípios e Comarcas"

# Salva
lu.comarcas.to_csv(
    path_or_buf=output_path_tab / f"{filename}.csv",
    index=False,
)
# lu.comarcas.to_excel(
#     excel_writer=output_path_tab / f"{filename}.xlsx",
#     sheet_name=f"{filename}",
#     index=False,
# )

<br>

---

## Análises das Comarcas


Inicialmente conferi se o nome dos Municípios definido pelo TJSP correspondem ao nome das Comarcas que levantei!

Felizmente, pelo que analisei, a resposta é sim!


In [ ]:
# Read Data
df_tjsp = pd.read_csv(
    filepath_or_buffer=output_path_tab / "Unidades, Municípios e Comarcas.csv",
)

# Aplica strip em todo o dataframe
#df_tjsp = df_tjsp.map(lambda x: x.strip() if isinstance(x, str) else x)
df_tjsp = lu.unidades

# Results
df_tjsp.info()
df_tjsp.head()

In [ ]:
# Filtra Colunas
df_tjsp = df_tjsp.drop(
    labels=[
        'unidades',
        'raj',
        #'id_municipio_tjsp'
    ],
    axis='columns',
    errors='ignore',
)

df_tjsp = df_tjsp.drop_duplicates()
df_tjsp_mun = df_tjsp.copy()

# Results
df_tjsp_mun.info()
df_tjsp_mun.head()

<br>

Crio uma tabela temporária, apenas para conseguir, posteriormente, trazer os códigos do IBGE para a tabela das comarcas


In [ ]:
# Comarca
df_tjsp_com = df_tjsp[df_tjsp['comarca_sede'] == 1]

# Filtra Colunas
# df_tjsp_com = df_tjsp_com.drop(
#     labels=[
#         #'municipio_corrigido',
#         #'comarca_sede',
#         #'municipio_tjsp'
#         'eee'
#     ],
#     axis='columns',
#     errors='ignore',
# )

# Results
df_tjsp_com.info()
df_tjsp_com.head()

In [ ]:
# Filtra Colunas
df_tjsp_com = df_tjsp_com.drop(
    labels=["comarca_sede"],
    axis="columns",
    errors="ignore",
)

#
df_tjsp_com = df_tjsp_com.drop_duplicates()


# Renomeia
df_tjsp_com = df_tjsp_com.rename(
    mapper={
        "municipio_tjsp_corrigido": "comarca_tjsp_corrigido",
        "id_municipio": "id_comarca",
    },
    axis=1,
)

# Deleta
df_tjsp_com = df_tjsp_com.drop(
    labels=["municipio_tjsp"],
    axis="columns",
    errors="ignore",
)

# Bata Bater
df_tjsp_com = adjust_columns(
    df=df_tjsp_com,
    column_ajust="comarca_tjsp",
)

In [ ]:
# Results
df_tjsp_com.info()
df_tjsp_com.head()

In [ ]:
df_script1 = pd.read_csv(
    filepath_or_buffer=output_path_tab / "Comarcas e CJs.csv",
)
display(df_script1.head())

# Aplica strip em todo o dataframe
# Já inclui
# df_script1 = df_script1.map(lambda x: x.strip() if isinstance(x, str) else x)

# Ajusta Coluna
df_script1 = adjust_columns(df=df_script1, column_ajust="comarca_tjsp")
display(df_script1.head())

Usando `left` eu dropo a Vila Mimosa!


In [ ]:
df_comarca = pd.merge(
    left=df_tjsp_com,
    right=df_script1,
    left_on='comarca_tjsp_temp',
    right_on='comarca_tjsp_temp',
    how='left',
    suffixes=['', '_copy'],
)

# Deleta
df_comarca = df_comarca.drop(
    labels=['comarca_tjsp_copy'],
    axis='columns',
    errors='ignore',
)

df_comarca

In [ ]:
df_comarca = df_comarca.iloc[
    df_comarca['comarca_tjsp'].str.normalize('NFKD').argsort()
]
df_comarca = df_comarca.reset_index(drop=True)
df_comarca.head()

In [ ]:
df_comarca[df_comarca['id_cj'].isna()]

In [ ]:
# df_comarca[df_comarca['id_comarca'].isna()]

In [ ]:
df_comarca = df_comarca.drop(
    labels=["comarca_tjsp_temp", "id_municipio_tjsp"],
    axis="columns",
)

# Results
df_comarca.info()
df_comarca.head()

In [ ]:
filename = "Comarcas"

# Salva
df_comarca.to_csv(
    path_or_buf=output_path_tab / f"{filename}.csv",
    index=False,
)
# df_comarca.to_excel(
#     excel_writer=output_path_tab / f"{filename}.xlsx",
#     sheet_name=f"{filename}",
#     index=False,
# )

In [ ]:
for file in list(output_path_tab.glob('Comarcas e CJs*')):
    print(file)
    file.unlink()

<br>

---

## Municípios

Adiciona à tabela dos municípios as comarcas


In [ ]:
# Read Data
df_tjsp = pd.read_csv(
    filepath_or_buffer=output_path_tab / "Unidades, Municípios e Comarcas.csv",
)

# Aplica strip em todo o dataframe
df_tjsp = df_tjsp.map(lambda x: x.strip() if isinstance(x, str) else x)

# Results
df_tjsp.info()
df_tjsp.head()

In [ ]:
# Filtra Colunas
df_tjsp = df_tjsp.drop(
    labels=[
        'unidades',
        'raj',
        'municipio_tjsp',
        'municipio_tjsp_corrigido',
        #'id_municipio_tjsp',
    ],
    axis='columns',
    errors='ignore',
)

df_tjsp = df_tjsp.drop_duplicates()
df_tjsp_mun = df_tjsp.copy()
# df_tjsp_mun[df_tjsp_mun['municipio_tjsp'] == 'Charqueada']
df_tjsp_mun

In [ ]:
# Ajusta Coluna
df_tjsp_mun = adjust_columns(
    df=df_tjsp_mun,
    # column_ajust="municipio_tjsp",
    column_ajust="municipio_tjsp_corrigido",
)

<br>

---

### Municípios


In [ ]:
df_script3 = pd.read_csv(filepath_or_buffer=output_path_tab / 'Municipios.csv')

# Aplica strip em todo o dataframe
df_script3 = df_script3.map(lambda x: x.strip() if isinstance(x, str) else x)

# Results
df_script3.info()
df_script3.head()

In [ ]:
# Ajusta Coluna
df_script3 = adjust_columns(df=df_script3, column_ajust='municipio_tjsp')

In [ ]:
# Merge
df_tjsp_merged = pd.merge(
    left=df_script3,
    right=df_tjsp_mun,
    left_on='municipio_tjsp_temp',
    right_on='municipio_tjsp_temp',
    how='left',
    suffixes=['', '_copy'],
)

# Results
df_tjsp_merged.info()
df_tjsp_merged.head()

In [ ]:
# Filtra Colunas
df_tjsp_merged = df_tjsp_merged.drop(
    labels=[
        'municipio_tjsp_temp',
        'id_municipio_tjsp_copy',
        'municipio_tjsp_copy',
        'id_municipio_copy',
    ],
    axis='columns',
    errors='ignore',
)

# Results
df_tjsp_merged.info()
df_tjsp_merged.head()

Uma vez que adicionei o nome da Comarca...\
Desejo converter para número...

Para isso eu leio a tabela de comarcas.


<br>

---

### Comarcas


In [ ]:
# Read Data
df_tjsp_com = pd.read_csv(filepath_or_buffer=output_path_tab / 'Comarcas.csv')

# Results
df_tjsp_com.info()
df_tjsp_com.head()

In [ ]:
df_tjsp_com = adjust_columns(df=df_tjsp_com, column_ajust='comarca_tjsp')
df_tjsp_com

In [ ]:
df_tjsp_merged = adjust_columns(df=df_tjsp_merged, column_ajust='comarca_tjsp')
df_tjsp_merged

In [ ]:
df_municipio = pd.merge(
    left=df_tjsp_merged,
    right=df_tjsp_com,
    left_on='comarca_tjsp_temp',
    right_on='comarca_tjsp_temp',
    how='left',
    suffixes=['', '_copy'],
)

# 
df_municipio = df_municipio.drop(
    labels=[
        'comarca_tjsp_temp',
        'comarca_tjsp_corrigido',
        'comarca_tjsp_copy',
        'comarca_tjsp',
    ],
    axis='columns',
)

# Results
df_municipio.info()
df_municipio.head()

In [ ]:
filename = "Municipios"

# Salva
df_municipio.to_csv(
    path_or_buf=output_path_tab / f"{filename}.csv",
    index=False,
)
df_municipio.to_excel(
    excel_writer=output_path_tab / f"{filename}.xlsx",
    sheet_name=f"{filename}",
    index=False,
)